# AirShift — Data Cleaning

This notebook prepares the raw air quality data for analysis and machine learning.

The cleaning process focuses on:

* Combining data from all monitoring stations
* Creating a consistent datetime column
* Validating data types and values
* Handling missing values carefully
* Checking duplicates and temporal consistency
* Saving the cleaned dataset for the next stage of the AirShift pipeline

No raw data will be overwritten during the cleaning process.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
# Project directories
DATA_DIR = Path("../data/raw")
OUTPUT_DIR = Path("../data/processed")

# Create processed-data directory if it does not exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Raw data directory:")
print(DATA_DIR.resolve())

print("\nProcessed data directory:")
print(OUTPUT_DIR.resolve())

Raw data directory:
C:\Users\HP\Desktop\Projects\airshift\data\raw

Processed data directory:
C:\Users\HP\Desktop\Projects\airshift\data\processed


In [3]:
files = sorted(DATA_DIR.glob("*.csv"))

print(f"Number of CSV files: {len(files)}")

for file in files:
    print(file.name)

Number of CSV files: 12
PRSA_Data_Aotizhongxin_20130301-20170228.csv
PRSA_Data_Changping_20130301-20170228.csv
PRSA_Data_Dingling_20130301-20170228.csv
PRSA_Data_Dongsi_20130301-20170228.csv
PRSA_Data_Guanyuan_20130301-20170228.csv
PRSA_Data_Gucheng_20130301-20170228.csv
PRSA_Data_Huairou_20130301-20170228.csv
PRSA_Data_Nongzhanguan_20130301-20170228.csv
PRSA_Data_Shunyi_20130301-20170228.csv
PRSA_Data_Tiantan_20130301-20170228.csv
PRSA_Data_Wanliu_20130301-20170228.csv
PRSA_Data_Wanshouxigong_20130301-20170228.csv


## 1. Load and Combine Station Data

The raw dataset contains hourly air quality measurements from 12 monitoring stations.

We combine the station files into a single dataset while preserving the `station` column. This provides one consistent table for subsequent cleaning and feature engineering.

The original raw files are kept unchanged.

In [4]:
station_data = []

for file in files:
    df = pd.read_csv(file)
    station_data.append(df)

df = pd.concat(station_data, ignore_index=True)

print("Combined dataset shape:", df.shape)
print("\nNumber of stations:", df["station"].nunique())
print("\nStations:")
print(sorted(df["station"].unique()))

Combined dataset shape: (420768, 18)

Number of stations: 12

Stations:
['Aotizhongxin', 'Changping', 'Dingling', 'Dongsi', 'Guanyuan', 'Gucheng', 'Huairou', 'Nongzhanguan', 'Shunyi', 'Tiantan', 'Wanliu', 'Wanshouxigong']


## 2. Create Datetime

The raw dataset stores time information in four separate columns: `year`, `month`, `day`, and `hour`.

We combine these columns into a single `datetime` column to provide a consistent representation of time.

This will make the data easier to work with during time-series analysis, temporal feature engineering, sorting, and the definition of historical and future conditions for the AirShift early-warning task.


In [5]:
df["datetime"] = pd.to_datetime(
    df[["year", "month", "day", "hour"]]
)

print("Datetime column created successfully.")

print("\nFirst timestamps:")
print(df["datetime"].head())

print("\nLast timestamps:")
print(df["datetime"].tail())

Datetime column created successfully.

First timestamps:
0   2013-03-01 00:00:00
1   2013-03-01 01:00:00
2   2013-03-01 02:00:00
3   2013-03-01 03:00:00
4   2013-03-01 04:00:00
Name: datetime, dtype: datetime64[us]

Last timestamps:
420763   2017-02-28 19:00:00
420764   2017-02-28 20:00:00
420765   2017-02-28 21:00:00
420766   2017-02-28 22:00:00
420767   2017-02-28 23:00:00
Name: datetime, dtype: datetime64[us]


## 3. Validate Data Types

Before handling missing values or invalid observations, we verify that each column has an appropriate data type.

Correct data types are important because they determine how the variables can be analyzed and processed during the cleaning and machine learning stages.

In particular, the temporal columns should be numeric, the measurement variables should be numeric, and the `station` and `wd` columns should be treated as categorical variables.


In [6]:
df.dtypes

No                   int64
year                 int64
month                int64
day                  int64
hour                 int64
PM2.5              float64
PM10               float64
SO2                float64
NO2                float64
CO                 float64
O3                 float64
TEMP               float64
PRES               float64
DEWP               float64
RAIN               float64
wd                     str
WSPM               float64
station                str
datetime    datetime64[us]
dtype: object

### Data Types Validation Findings

The data types are appropriate for the current cleaning stage.

* Temporal components (`year`, `month`, `day`, and `hour`) are stored as integers.
* Air quality and meteorological measurements are stored as numeric (`float64`) variables.
* `wd` and `station` are stored as string variables and will be treated as categorical features when needed.
* The newly created `datetime` column is stored as a datetime type suitable for time-series operations.

No data type conversions are required at this stage.

## 4. Check for Invalid Values

After validating the data types, we check whether the measurement variables contain physically or logically invalid values.

This step is important because a value can have a correct data type while still being invalid. For example, negative rainfall or negative wind speed would not be physically meaningful.

The following checks focus on variables for which clear non-negative constraints can be applied. Potentially extreme but physically possible values will not be removed automatically, as high pollution levels may represent genuine air quality deterioration events that are important for the AirShift early-warning task.


In [7]:
non_negative_columns = [
    "PM2.5",
    "PM10",
    "SO2",
    "NO2",
    "CO",
    "O3",
    "PRES",
    "RAIN",
    "WSPM"
]

invalid_values = {}

for column in non_negative_columns:
    invalid_count = (df[column] < 0).sum()
    invalid_values[column] = invalid_count

invalid_values_df = pd.DataFrame(
    list(invalid_values.items()),
    columns=["feature", "negative_values"]
)

invalid_values_df

,feature,negative_values
0,PM2.5,0
1,PM10,0
2,SO2,0
3,NO2,0
4,CO,0
5,O3,0
6,PRES,0
7,RAIN,0
8,WSPM,0


### Invalid Values Findings

The validation found no negative values in the variables with non-negative physical constraints.

All checked pollutant concentrations (`PM2.5`, `PM10`, `SO2`, `NO2`, `CO`, and `O3`), as well as `PRES`, `RAIN`, and `WSPM`, contain zero negative observations.

Negative values were not considered invalid for temperature-related variables such as `TEMP` and `DEWP`, because sub-zero values are physically possible and represent realistic environmental conditions.

Therefore, no invalid negative values were identified in this check, and no observations were removed at this stage.


## 5. Missing Values

Missing values were identified during the initial data profiling. Before applying any treatment, we reassess missing values in the combined dataset to determine their overall distribution across features and monitoring stations.

Because AirShift is a time-series early-warning system, missing values will not be removed or imputed blindly. The treatment strategy will consider both the amount of missing data and the duration of consecutive missing periods.


In [8]:
missing_summary = (
    df.isnull()
    .sum()
    .reset_index()
)

missing_summary.columns = ["feature", "missing_count"]

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] / len(df) * 100
).round(2)

missing_summary = (
    missing_summary[missing_summary["missing_count"] > 0]
    .sort_values("missing_count", ascending=False)
)

missing_summary

,feature,missing_count,missing_percentage
9,CO,20701,4.92
10,O3,13277,3.16
8,NO2,12116,2.88
7,SO2,9021,2.14
5,PM2.5,8739,2.08
6,PM10,6449,1.53
15,wd,1822,0.43
13,DEWP,403,0.10
11,TEMP,398,0.09
12,PRES,393,0.09


In [9]:
missing_summary = (
    df.isnull()
    .sum()
    .reset_index()
)

missing_summary.columns = ["feature", "missing_count"]

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] / len(df) * 100
).round(2)

missing_summary = (
    missing_summary[missing_summary["missing_count"] > 0]
    .sort_values("missing_count", ascending=False)
)

missing_summary

,feature,missing_count,missing_percentage
9,CO,20701,4.92
10,O3,13277,3.16
8,NO2,12116,2.88
7,SO2,9021,2.14
5,PM2.5,8739,2.08
6,PM10,6449,1.53
15,wd,1822,0.43
13,DEWP,403,0.10
11,TEMP,398,0.09
12,PRES,393,0.09


### Missing Values Findings

The combined dataset contains missing values mainly in air pollutant measurements. `CO` has the highest missing rate at 4.92%, followed by `O3` (3.16%) and `NO2` (2.88%). The remaining major pollutant variables have missing rates between 1.53% and 2.14%.

Most meteorological variables have very low missing rates, below 0.10%, while `wd` has a slightly higher missing rate of 0.43%.

Although the overall proportion of missing values is relatively low, the earlier temporal analysis showed that some missing values occur in long consecutive blocks. Therefore, missing values will be handled based on both their proportion and temporal distribution rather than using a single treatment method for all variables.


## 6. Analyze Missing Values by Row

Missing values can occur in individual observations or across multiple variables within the same observation.

Before selecting an imputation or removal strategy, we examine how many features are missing in each row. This helps determine whether missing observations are isolated or whether some timestamps contain substantial amounts of missing information.

Because AirShift relies on temporal patterns, rows will not be removed automatically based only on the presence of missing values.


In [10]:
missing_per_row = df.isnull().sum(axis=1)

row_missing_summary = (
    missing_per_row
    .value_counts()
    .sort_index()
    .reset_index()
)

row_missing_summary.columns = [
    "missing_features",
    "number_of_rows"
]

row_missing_summary

,missing_features,number_of_rows
0,0,382168
1,1,27203
2,2,4571
3,3,445
4,4,862
5,5,241
6,6,5252
7,7,26


### 6.1 Inspect Rows with Extensive Missingness

The row-level analysis shows that most observations contain complete measurements, while a smaller number of observations contain multiple missing features.

In particular, some rows contain six or seven missing features simultaneously. These observations require further inspection to determine which variables are missing and whether the missing values are concentrated within specific measurement groups.

No rows will be removed at this stage. Further analysis is required before deciding how observations with extensive missingness should be handled.


### 6.2 Inspect Rows with Extensive Missingness

The row-level analysis identified a small number of observations with multiple missing features. In particular, some rows contain six or seven missing values simultaneously.

We further inspect these observations to identify which features are missing and determine whether the missing values are concentrated in specific variables.

This analysis helps us decide whether these observations can be retained, require imputation, or should be excluded during the data cleaning process.


In [11]:
rows_with_many_missing = df.loc[
    missing_per_row >= 6
]

print("Rows with 6 or more missing features:")
print(rows_with_many_missing.shape)

print("\nMissing values by feature:")
print(
    rows_with_many_missing.isnull().sum()
    .sort_values(ascending=False)
)

Rows with 6 or more missing features:
(5278, 19)

Missing values by feature:
O3          4974
PM10        4971
CO          4970
NO2         4969
SO2         4969
PM2.5       4969
wd           327
RAIN         309
WSPM         309
DEWP         309
TEMP         309
PRES         309
month          0
No             0
year           0
hour           0
day            0
station        0
datetime       0
dtype: int64


### Extensive Missingness Findings

The inspection of observations with six or more missing features shows that extensive missingness is concentrated primarily in air pollutant measurements.

Among these observations, `O3`, `PM10`, `CO`, `NO2`, `SO2`, and `PM2.5` account for most of the missing values. Meteorological variables have considerably fewer missing observations within these rows, while `wd` also shows a relatively small number of missing values.

This pattern suggests that extensive missingness is mainly associated with pollutant measurements rather than affecting all variables equally. Therefore, these observations should not be removed automatically. Their temporal distribution and relevance to the target variables will be considered when defining the missing-data treatment strategy.


### 6.3 Missing Values by Station

Missing values may not be distributed equally across all monitoring stations.

We therefore examine the amount and percentage of missing values at each station. This helps identify whether some stations have substantially higher missingness than others.

Understanding the distribution of missing values across stations is important before selecting an appropriate treatment strategy.


In [12]:
measurement_columns = [
    "PM2.5",
    "PM10",
    "SO2",
    "NO2",
    "CO",
    "O3",
    "TEMP",
    "PRES",
    "DEWP",
    "RAIN",
    "wd",
    "WSPM"
]

station_missing = (
    df.groupby("station")[measurement_columns]
      .apply(lambda x: x.isnull().sum().sum())
      .reset_index(name="missing_values")
)

station_rows = df.groupby("station").size().reset_index(name="rows")

station_missing = station_missing.merge(
    station_rows,
    on="station"
)

station_missing["missing_percentage"] = (
    station_missing["missing_values"]
    / (station_missing["rows"] * len(measurement_columns))
    * 100
).round(2)

station_missing.sort_values(
    "missing_percentage",
    ascending=False
)

,station,missing_values,rows,missing_percentage
8,Shunyi,8523,35064,2.03
3,Dongsi,7600,35064,1.81
6,Huairou,7485,35064,1.78
0,Aotizhongxin,7271,35064,1.73
2,Dingling,7015,35064,1.67
10,Wanliu,6447,35064,1.53
9,Tiantan,5277,35064,1.25
4,Guanyuan,5279,35064,1.25
1,Changping,5166,35064,1.23
11,Wanshouxigong,5146,35064,1.22


### Missing Values by Station Findings

The distribution of missing values varies across the monitoring stations, but the differences are relatively moderate.

`Shunyi` has the highest overall missing rate at 2.03%, followed by `Dongsi` at 1.81% and `Huairou` at 1.78%. `Nongzhanguan` has the lowest missing rate at 0.97%.

Although some stations have higher missingness than others, no station shows an exceptionally high missing rate that would justify excluding the entire station from the dataset.

Therefore, missing values will be handled at the feature and temporal level rather than removing stations based solely on their overall missingness.


## 7. Missing Data Treatment Strategy

Missing values will be handled based on their type, amount, and temporal pattern.

* Short gaps in numerical variables may be interpolated.
* Long gaps will not be filled using simple interpolation.
* `wd` will be handled separately as a categorical variable.
* Rows with extensive missingness will not be removed automatically.
* Valid extreme pollution values will be retained.


In [13]:
numeric_columns = [
    "PM2.5",
    "PM10",
    "SO2",
    "NO2",
    "CO",
    "O3",
    "TEMP",
    "PRES",
    "DEWP",
    "RAIN",
    "WSPM"
]

df = df.sort_values(["station", "datetime"]).copy()

df[numeric_columns] = (
    df.groupby("station")[numeric_columns]
      .transform(
          lambda x: x.interpolate(
              method="linear",
              limit=6,
              limit_direction="both"
          )
      )
)

### 7.1 Define Missing Data Treatment Strategy

Missing values will be handled based on their type, amount, and temporal pattern.

* Short gaps in numerical variables may be interpolated.
* Long gaps will not be filled using simple interpolation.
* `wd` will be handled separately as a categorical variable.
* Rows with extensive missingness will not be removed automatically.
* Valid extreme pollution values will be retained.


### 7.2 Prepare Data for Short-Gap Interpolation

Short-gap interpolation will be applied only to numerical variables and within each monitoring station.

Before interpolation, the data is sorted chronologically to ensure that missing values are treated according to the correct temporal order.


In [14]:
numeric_columns = [
    "PM2.5",
    "PM10",
    "SO2",
    "NO2",
    "CO",
    "O3",
    "TEMP",
    "PRES",
    "DEWP",
    "RAIN",
    "WSPM"
]

df = df.sort_values(
    ["station", "datetime"]
).copy()

print("Numerical columns:")
print(numeric_columns)

print("\nData sorted by station and datetime.")

Numerical columns:
['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']

Data sorted by station and datetime.


### 7.3 Interpolate Short Missing Gaps

Short missing gaps in numerical variables are filled using linear interpolation within each monitoring station.

Only gaps of up to 6 consecutive hours are interpolated. Longer gaps are preserved as missing values to avoid creating unrealistic values across extended periods without observations.


In [15]:
df_before_interpolation = df.copy()

df[numeric_columns] = (
    df.groupby("station")[numeric_columns]
      .transform(
          lambda x: x.interpolate(
              method="linear",
              limit=6,
              limit_area="inside"
          )
      )
)

print("Short-gap interpolation completed.")

Short-gap interpolation completed.


### 7.4 Validate Interpolation

After interpolation, the number of missing values before and after treatment is compared.

This verifies that short gaps were filled while longer missing periods were preserved.


In [16]:
missing_before = (
    df_before_interpolation[numeric_columns]
    .isnull()
    .sum()
)

missing_after = (
    df[numeric_columns]
    .isnull()
    .sum()
)

interpolation_summary = pd.DataFrame({
    "missing_before": missing_before,
    "missing_after": missing_after,
    "values_interpolated": missing_before - missing_after
})

interpolation_summary

,missing_before,missing_after,values_interpolated
PM2.5,2471,1877,594
PM10,1653,1247,406
SO2,2977,2390,587
NO2,4467,3781,686
CO,10458,9499,959
O3,4028,3254,774
TEMP,0,0,0
PRES,0,0,0
DEWP,0,0,0
RAIN,0,0,0


### Interpolation Findings

Short-gap interpolation reduced missing values in the pollutant variables.

A total of 4,006 missing values were interpolated across `PM2.5`, `PM10`, `SO2`, `NO2`, `CO`, and `O3`.

Remaining missing values correspond mainly to longer gaps that were not suitable for interpolation.


### 7.5 Analyze Remaining Missing Values

After short-gap interpolation, some missing values remain, mainly due to longer gaps or missing values at the boundaries of the time series.

These remaining values are analyzed before deciding how they should be handled.


In [17]:
remaining_missing = (
    df.isnull()
      .sum()
      .sort_values(ascending=False)
)

remaining_missing = remaining_missing[
    remaining_missing > 0
]

remaining_missing

CO       9499
NO2      3781
O3       3254
SO2      2390
PM2.5    1877
wd       1822
PM10     1247
dtype: int64

### 7.6 Analyze Remaining Missing Gaps

The remaining missing values are examined by measuring the length of consecutive missing periods for each affected variable.

This helps distinguish short gaps from prolonged gaps and supports the decision on how the remaining missing values should be handled.


In [18]:
remaining_missing_columns = [
    "PM2.5",
    "PM10",
    "SO2",
    "NO2",
    "CO",
    "O3",
    "wd"
]

gap_summary = []

for column in remaining_missing_columns:
    missing = df[column].isnull()

    groups = (missing != missing.shift()).cumsum()
    gap_lengths = missing.groupby(groups).sum()

    gap_lengths = gap_lengths[gap_lengths > 0]

    gap_summary.append({
        "feature": column,
        "number_of_gaps": len(gap_lengths),
        "longest_gap_hours": gap_lengths.max()
    })

gap_summary_df = pd.DataFrame(gap_summary)

gap_summary_df.sort_values(
    "longest_gap_hours",
    ascending=False
)

,feature,number_of_gaps,longest_gap_hours
4,CO,115,1499
3,NO2,84,882
5,O3,82,479
2,SO2,74,439
0,PM2.5,73,325
1,PM10,46,325
6,wd,1439,13


### Remaining Missing Gaps Findings

The remaining missing values are mainly associated with prolonged gaps in pollutant measurements.

`CO` has the longest remaining gap at 1,499 hours, followed by `NO2` at 882 hours and `O3` at 479 hours. In contrast, `wd` has many short gaps, with a maximum gap of 13 hours.

These results confirm that the remaining pollutant missing values should not be filled using simple interpolation.


### 7.7 Handle Remaining Long Gaps

Long missing gaps in pollutant measurements are retained as missing values rather than filled using simple interpolation.

This avoids introducing artificial pollution patterns that could affect the AirShift early-warning model.


In [19]:
long_gap_missing = (
    df[remaining_missing_columns]
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

long_gap_missing

CO       9499
NO2      3781
O3       3254
SO2      2390
PM2.5    1877
wd       1822
PM10     1247
dtype: int64

### 7.8 Handle Missing Wind Direction

Missing values in `wd` are handled separately because wind direction is a categorical variable.

Since the remaining gaps are relatively short, forward filling is used within each monitoring station to preserve the temporal structure of the data.


In [20]:
df["wd"] = (
    df.groupby("station")["wd"]
      .ffill()
)

print("Remaining missing values in wd:", df["wd"].isnull().sum())

Remaining missing values in wd: 0


### 7.9 Analyze Rows with Remaining Missing Values

After handling missing wind direction values, the remaining missing values are limited to pollutant measurements.

The number of affected rows is examined before deciding whether any observations should be removed.


In [21]:
pollutant_columns = [
    "PM2.5",
    "PM10",
    "SO2",
    "NO2",
    "CO",
    "O3"
]

rows_with_remaining_missing = df[
    df[pollutant_columns].isnull().any(axis=1)
]

print("Rows with remaining pollutant missing values:",
      len(rows_with_remaining_missing))

print("\nPercentage of dataset:")
print(
    round(
        len(rows_with_remaining_missing) / len(df) * 100,
        2
    ),
    "%"
)

Rows with remaining pollutant missing values: 14654

Percentage of dataset:
3.48 %


### Remaining Missing Values Findings

After the missing wind direction values were handled, 14,654 rows (3.48% of the dataset) still contain missing pollutant measurements.

These missing values are mainly associated with prolonged gaps. Since the proportion is relatively small and the data has a temporal structure, these rows will not be removed automatically.


### 7.10 Finalize Missing Value Treatment

After the missing-value analysis and treatment, short numerical gaps have been interpolated, missing `wd` values have been filled, and long pollutant gaps have been retained as missing.

Rows with remaining pollutant missing values will be handled later during feature engineering and model preparation, based on the requirements of the prediction task.


## 8. Validate Cleaned Dataset

After completing the cleaning process, the dataset is validated to ensure that its structure and temporal consistency have been preserved.

The validation focuses on:

* Checking for duplicate rows
* Verifying temporal consistency within each station
* Reviewing the final missing values
* Confirming the final dataset structure

No additional data treatment is performed during this stage.


### 8.1 Check for Duplicates

Duplicate observations can affect analysis and machine learning results.

We check the cleaned dataset for duplicate rows to ensure that each observation is represented only once.


In [22]:
duplicate_rows = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_rows)

Number of duplicate rows: 0


### 8.2 Check Temporal Consistency

Because AirShift is based on time-series data, we verify that observations are correctly ordered within each monitoring station.

The check ensures that timestamps are sorted chronologically and that no duplicate timestamps exist within a station.


In [23]:
temporal_check = df.groupby("station")["datetime"].agg(
    first_timestamp="min",
    last_timestamp="max",
    unique_timestamps="nunique",
    total_rows="size"
)

temporal_check["duplicate_timestamps"] = (
    temporal_check["total_rows"]
    - temporal_check["unique_timestamps"]
)

temporal_check

,first_timestamp,last_timestamp,unique_timestamps,total_rows,duplicate_timestamps
station,,,,,
Aotizhongxin,2013-03-01,2017-02-28 23:00:00,35064,35064,0
Changping,2013-03-01,2017-02-28 23:00:00,35064,35064,0
Dingling,2013-03-01,2017-02-28 23:00:00,35064,35064,0
Dongsi,2013-03-01,2017-02-28 23:00:00,35064,35064,0
Guanyuan,2013-03-01,2017-02-28 23:00:00,35064,35064,0
Gucheng,2013-03-01,2017-02-28 23:00:00,35064,35064,0
Huairou,2013-03-01,2017-02-28 23:00:00,35064,35064,0
Nongzhanguan,2013-03-01,2017-02-28 23:00:00,35064,35064,0
Shunyi,2013-03-01,2017-02-28 23:00:00,35064,35064,0


In [24]:
is_sorted = (
    df.groupby("station")["datetime"]
      .apply(lambda x: x.is_monotonic_increasing)
)

is_sorted

station
Aotizhongxin     True
Changping        True
Dingling         True
Dongsi           True
Guanyuan         True
Gucheng          True
Huairou          True
Nongzhanguan     True
Shunyi           True
Tiantan          True
Wanliu           True
Wanshouxigong    True
Name: datetime, dtype: bool